
MÁSTER BIG DATA & DATA ENGINEERING

Programación avanzada en Python

Autor: Yolanda García Ruiz



# Trabajo final

## Nota Preliminar

Antes de desarrollar el proyecto, es aconsejable leer el enunciado completamente, incluyendo las instrucciones de entrega y otros consejos que se  recogen al final de este enunciado.
En este enunciado se pueden producir cambios durante su desarrollo por distintos motivos: porque se descubre una situación anómala que no ha sido prevista al preparar el enunciado, porque algún estudiante encuentre un error o ambigüedad en la descripción o en los datos de partida o por cualquier otra causa. Esto ocurre también en cualquier proyecto real, así que tomárselo con el mejor talante y aprender también de estas eventuales contingencias. Se procurará informar debidamente del mejor modo posible cuando esto se produzca.


## Introducción

En el portal de datos abiertos del Ayuntamiento de Madrid se puede encontrar conjuntos de datos sobre multas de circulación que el Ayuntamiento de Madrid tramita cada mes, con todo el detalle posible sobre cada una de ellas que permite la legislación de protección de datos. 


La dirección del portal mencionado es la siguiente:
[https://datos.madrid.es](https://datos.madrid.es)


y el enlace donde se encuentran los datos relativos al detalle de las multas de dirculación  es el siguiente:

[https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/downloads](https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/downloads)


En éste último enlace  podemos encontrar los datos de multas por meses desde el año 2014. Se trata de un conjunto de URLs a ficheros en formato `csv`.
En esta práctica __solo nos va a interesar los datos desde junio el 2017__. 

### Metadatos

En cada uno de los ficheros `csv` podemos encontrar los siguientes datos:


| **Campo**         | **Descripción**                                                                                                                         |
| :---------------- | :-------------------------------------------------------------------------------------------------------------------------------------- |
| **CALIFICACION**  | Gravedad de la infracción cometida.                                                |
| **LUGAR**         | Dirección donde se cometió la infracción (calle y número).                                                                       |
| **MES**           | Mes en el que se registró la infracción.                                                                          |
| **ANIO**          | Año en que ocurrió la infracción (por ejemplo, 2019).                                                                                   |
| **HORA**          | Hora del día en la que se registró la infracción (en formato decimal, ej. `10.42`).                                                     |
| **IMP\_BOL**      | Importe de la sanción (en euros).                                                                                             |
| **DESCUENTO**     | Indica si aplica descuento por pronto pago.                                                                           |
| **PUNTOS**        | Número de puntos retirados del carnet de conducir (si aplica).                                                                          |
| **DENUNCIANTE**   | Autoridad o entidad que impuso la sanción.                                               |
| **HECHO-BOL**     | Descripción textual del hecho sancionado. |
| **VEL\_LIMITE**   | Límite de velocidad permitido (en km/h).                                                                |
| **VEL\_CIRCULA**  | Velocidad real a la que circulaba el vehículo (en km/h).                                                      |
| **COORDENADA\_X** | Coordenada X para geolocalizar la infracción.                                                          |
| **COORDENADA\_Y** | Coordenada Y para geolocalización.                                                                          |




## Librerías

Importar todas las librerías necesarias al principio, tal como propone el estilo `pep-8`. Ej.:
<a href="https://www.python.org/dev/peps/pep-0008/">PEP 8 -- Style Guide for Python Code</a>.

In [1]:
# Sol
import numpy as np
import pandas as pd
import requests
import re
from os import listdir
import urllib
import io
from io import StringIO
import matplotlib.pyplot as plt


## ETAPA 1. Análisis exploratorio y consultas.



### La descarga de datos

En este notebook vamos a estudiar únicamente los datos de diciembre  de 2024.
La URL que permite descargar dichos datos es la siguiente:

In [2]:
raiz = "https://datos.madrid.es/"
url_diciembre_24 = "dataset/210104-0-multas-circulacion-detalle/resource/210104-15-multas-circulacion-detalle-csv/download/210104-15-multas-circulacion-detalle-csv.csv" 
url = raiz + url_diciembre_24.lstrip("/")


Podemos hacer la petición con `request.get()`:

In [3]:
response = requests.get(url)
if response.status_code == 200:
    print(response.url)

https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/resource/210104-15-multas-circulacion-detalle-csv/download/210104-15-multas-circulacion-detalle-csv.csv


In [4]:
# Accedemos al texto 
content = io.StringIO(response.text)
multas = pd.read_csv(content, sep =';', encoding = 'latin1')
multas.head()

,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
0,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACI�N NO V�LIDA. ...,,,,
1,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACI�...",,,,
2,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACI�N DE UN...,,,,
3,LEVE,CL BRAVO MURILLO 24,12,2024,16.30,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACI�N NO V�LIDA. ...,,,,
4,LEVE,CL BRAVO MURILLO 16,12,2024,16.50,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACI�...",,,,


### Explorar la estructura de los datos en el dataframe

El método `info` devuelve información acerca de la estructura de un dataframe. Ejecuta el método `info()` sobre el dataframe `multas` y comenta los resultados.
Fíjate en las columnas que representan la hora, el día y el año y observa su tipo de datos. ¿Cuántas filas se han cargado?. Fíjate en los valores distintos de `NaN` en cada una de las columnas.

In [5]:
# Solución
multas.info()

<class 'pandas.DataFrame'>
RangeIndex: 249801 entries, 0 to 249800
Data columns (total 14 columns):
 #   Column                                                                                                                                                  Non-Null Count   Dtype  
---  ------                                                                                                                                                  --------------   -----  
 0   CALIFICACION                                                                                                                                            249801 non-null  str    
 1   LUGAR                                                                                                                                                   249801 non-null  str    
 2   MES                                                                                                                                                     249801 non-null

🔶 **Cesar Alejandro - Comentario:**
- Hay un total de 249801 filas, de las cuales a priori no hay ningún dato NaN o NULL. Pero esto es engañoso ya que columnas como VEL_LIMITE, VEL_CIRCULA y COORDENADA X/Y son strings (deberían ser numéricas) que los huecos vienen como espacios en blanco, y pandas los cuenta como no nulos.
- Respecto a la columna hora se ve que es un float donde se representa por ejemplo las 15:41 como 15.41.
- La columna día no existe, pero tenemos el mes, el cual es un dato de tipo entero así mismo como el anio.
- La columna 7 -> PUNTOS tiene un espacio al principio, lo cual hay que tener en cuenta.

###  Exploración de las columnas.

__Se pide:__

__a)__ ¿Cuántos valores distintos hay en la columna `CALIFICACION` del dataframe `multas`? Modifica los valores de dicha columna para que no contengan espacios en blanco. Puedes usar el método  `str.strip()` de la clase Series.

__b)__ ¿Cuántos valores distintos hay en las columnas `DESCUENTO`, `HECHO-BOL` y `DENUNCIANTE` del dataframe `multas`? Modifica los valores de dichas columnas para que no contengan espacios en blanco.

__c)__ Muestra el listado de columnas del dataframe.
* ¿Cuál es el nombre de la columna que representa la velocidad de circulación registrada? Fíjate en el espacio en blanco al final.
* ¿Cuál es el nombre de la columna que representa la coordenada Y de la ubicación geográfica de donde se produce la infracción? Fíjate en la cantidad de espacios en blanco al final.
* Muestra la serie de datos correspondiente a la coordenada Y.
* Usa el método  `rename` de los dataframes para normalizar los nombres de las columnas del dataframe multas, de forma que los nuevos nombres no tengan espacios en blanco al final. Consulta la ayuda del método `rename`; ésta te propone el uso de métodos de la clase `str` a cada elemento del índice de las columnas (o de las filas). En nuestro caso, para eliminar espacios en blanco usaremos el método `str.strip()` sobre el índice de las columnas.
  
__d)__ ¿Observa las columnas relativas a la velocidad. ¿De qué tipo son?. 
¿Cuántos valores distintos hay en las columnas `VEL_LIMITE`, `VEL_CIRCULA`. Por último, estudia la función `pd.to_numeric` de pandas y úsala para transformar el tipo de las columnas  anteriores a tipo numérico.

__e)__ ¿Observa las columnas relativas a la geolocalización. ¿De qué tipo son?. Usa la función `pd.to_numeric` de pandas para transformar el tipo de las columnas  `COORDENADA-X`, `COORDENADA-Y` a tipo numérico.

El dataframe obtenido después de aplicar todas las transformaciones anteriores 

In [6]:
# Solución

# Se crea un DF a partir de multas, esto es para no editar el original en caso de que haga falta recuperarlo o usarlo para otro motivo que aun no esté tratado.
clean_multas = multas.copy()

# Ejercicio a.
print("Valores distintos antes de strip:", clean_multas["CALIFICACION"].nunique())
clean_multas["CALIFICACION"] = clean_multas["CALIFICACION"].str.strip()
print("Valores distintos despues de strip:", clean_multas["CALIFICACION"].nunique())

Valores distintos antes de strip: 3
Valores distintos despues de strip: 3


🔶 **Cesar Alejandro - Comentario:**
- Se observan 3 valores distintos antes y después del strip, los cuales son LEVE, GRAVE y MUY GRAVE.

In [7]:
# Ejercicio b.
columnas = ["DESCUENTO","HECHO-BOL","DENUNCIANTE"]
print("Valores distintos antes de strip:", clean_multas[columnas].nunique())
print("============================================")
clean_multas[columnas] = clean_multas[columnas].map(str.strip)
print("Valores distintos despues de strip:", clean_multas[columnas].nunique())

Valores distintos antes de strip: DESCUENTO        1
HECHO-BOL      345
DENUNCIANTE      6
dtype: int64
Valores distintos despues de strip: DESCUENTO        1
HECHO-BOL      345
DENUNCIANTE      6
dtype: int64


🔶 **Cesar Alejandro - Comentario:**
- Las 3 columnas tienen lo mismo antes y despues del strip: 1, 345 y 6 valores.

In [8]:
# Ejercicio c.
print("Columnas sin limpiar\n", clean_multas.columns)
print("============================================")
print("Columna que registra la velocidad de circulación: ", clean_multas.columns[11])
print("Columna que registra la coordenada de geolocalización Y: ", clean_multas.columns[-1])
print("============================================")
print(clean_multas[clean_multas.columns[-1]])
print("============================================")
#pd.DataFrame.rename?
clean_multas = clean_multas.rename(columns=str.strip)
print("Columnas limpiadas\n", clean_multas.columns)

Columnas sin limpiar
 Index(['CALIFICACION', 'LUGAR', 'MES', 'ANIO', 'HORA', 'IMP_BOL', 'DESCUENTO',
       ' PUNTOS', 'DENUNCIANTE', 'HECHO-BOL', 'VEL_LIMITE', 'VEL_CIRCULA ',
       'COORDENADA-X',
       'COORDENADA-Y                                                                                                                                          '],
      dtype='str')
Columna que registra la velocidad de circulación:  VEL_CIRCULA 
Columna que registra la coordenada de geolocalización Y:  COORDENADA-Y                                                                                                                                          
0                    
1                    
2                    
3                    
4                    
             ...     
249796               
249797               
249798               
249799               
249800               
Name: COORDENADA-Y                                                                                      

🔶 **Cesar Alejandro - Comentario:**
- Se puede observar que PUNTOS, VEL_CIRCULA y COORDENADA-Y tenian espacios en blanco.
- La función rename(columns = str.strip) es aplicada a cada nombre de la columna.

In [9]:
# Ejercicio d.
columnas = ["VEL_LIMITE","VEL_CIRCULA"]
print(clean_multas[columnas].dtypes)
print("============================================")
print("Valores distintos\n", clean_multas[columnas].nunique())
print("Valores distintos VEL_LIMITE: ", clean_multas["VEL_LIMITE"].unique())
print("Valores distintos VEL_CIRCULA:", clean_multas["VEL_CIRCULA"].unique())
print("============================================")
clean_multas[columnas] = clean_multas[columnas].apply(pd.to_numeric, errors="coerce")
print("Valores distintos\n", clean_multas[columnas].nunique())
print("Valores distintos VEL_LIMITE: ", clean_multas["VEL_LIMITE"].unique())
print("Valores distintos VEL_CIRCULA:", clean_multas["VEL_CIRCULA"].unique())

VEL_LIMITE     str
VEL_CIRCULA    str
dtype: object
Valores distintos
 VEL_LIMITE       7
VEL_CIRCULA    101
dtype: int64
Valores distintos VEL_LIMITE:  <StringArray>
['   ', ' 50', ' 60', ' 40', ' 30', ' 90', ' 70']
Length: 7, dtype: str
Valores distintos VEL_CIRCULA: <StringArray>
['   ', ' 74', ' 79', '100', ' 73', '103', ' 96', ' 91', ' 77', ' 95',
 ...
 '125', '138', '142', '128', '134', '141', '130', '135', '137', '139']
Length: 101, dtype: str
Valores distintos
 VEL_LIMITE       6
VEL_CIRCULA    100
dtype: int64
Valores distintos VEL_LIMITE:  [nan 50. 60. 40. 30. 90. 70.]
Valores distintos VEL_CIRCULA: [ nan  74.  79. 100.  73. 103.  96.  91.  77.  95.  94. 104.  75.  82.
 109.  83.  84.  81. 114.  71.  97.  78. 110.  72.  87.  65. 106.  58.
  88.  99. 102.  52.  68.  86.  85.  80.  76.  66.  67.  62.  61. 111.
  93.  69.  98. 105. 108. 107.  92. 101. 122. 123.  54. 157. 124.  53.
  59. 131. 129. 126. 120. 115. 118.  90.  89.  63.  64.  70. 113. 117.
 112. 116. 127. 132.  57.  6

🔶 **Cesar Alejandro - Comentario:**
- Se ven que son de tipo STR ya que pandas no puede leer la columna como número al traer espacios en blanco.
- Al hacer el conteo con NUNIQUE se puede ver que reduce en 1, ya que NaN no cuenta.
- Con coerce en to_numeric le decimos como tratar los errores y que queremos que haga con ellos, en este caso coerce los transforma a nulos.

In [10]:
# Ejercicio e.
columnas = ["COORDENADA-X", "COORDENADA-Y"]
print(clean_multas[columnas].dtypes)
print("============================================")
print("Valores distintos\n", clean_multas[columnas].nunique())
print("Valores distintos COORDENADA-X: ", clean_multas["COORDENADA-X"].unique())
print("Valores distintos COORDENADA-Y:", clean_multas["COORDENADA-Y"].unique())
print("============================================")
clean_multas[columnas] = clean_multas[columnas].apply(pd.to_numeric, errors="coerce")
print("Valores distintos\n", clean_multas[columnas].nunique())
print("Valores distintos COORDENADA-X: ", clean_multas["COORDENADA-X"].unique())
print("Valores distintos COORDENADA-Y:", clean_multas["COORDENADA-Y"].unique())

COORDENADA-X    str
COORDENADA-Y    str
dtype: object
Valores distintos
 COORDENADA-X    2455
COORDENADA-Y    2500
dtype: int64
Valores distintos COORDENADA-X:  <StringArray>
['           ', '    4407.67', '    4397.66', '    4379.90', '    4400.43',
 '    4400.45', '    4380.20', '    4376.66', '    4400.75', '    4400.76',
 ...
 '  440969.81', '  440932.89', '  443567.10', '  438625.72', '  443343.14',
 '  439415.77', '  443917.13', '  438327.34', '  443921.94', '  444180.30']
Length: 2455, dtype: str
Valores distintos COORDENADA-Y: <StringArray>
['           ', '   44748.29', '   44751.88', '   44743.71', '   44736.05',
 '   44743.51', '   44746.35', '   44746.39', '   44748.87', '   44748.92',
 ...
 ' 4471455.18', ' 4471961.66', ' 4473383.23', ' 4475823.04', ' 4478976.86',
 ' 4472459.08', ' 4476649.40', ' 4475117.81', ' 4476601.18', ' 4482686.90']
Length: 2500, dtype: str
Valores distintos
 COORDENADA-X    2454
COORDENADA-Y    2499
dtype: int64
Valores distintos COORDENADA-X:  [   

🔶 **Cesar Alejandro - Comentario:**
- Como el ejemplo anterior, se transforma de STR a un FLOAT. Se ve que aproximadamente la mitad de las multas no tienen coordenadas.

### Creación de columna nueva

__a)__ A partir del dataframe `multas` que contiene las columnas `MES`, `ANIO` y `HORA`, crear una nueva columna llamada `fecha` de tipo datetime que combine dicha información en un único objeto de fecha y hora. Usa como día el día 1.

__b)__ Modifica el dataframe `multas` de forma que el índice sea el valor de la columna creada anteriormente.

__c)__ Elimina del dataframe la columna `fecha`.



In [11]:
# solución
hr = (clean_multas["HORA"] * 100).round().astype(int)
fecha = pd.DataFrame({
    "year": clean_multas["ANIO"],
    "month": clean_multas["MES"],
    "day": 1,
    "hour": hr // 100,
    "minute": hr % 100
})
clean_multas["FECHA"] = pd.to_datetime(fecha)
print(clean_multas["FECHA"].dtype)
clean_multas = clean_multas.set_index("FECHA")
clean_multas = clean_multas.sort_index()
clean_multas

datetime64[us]


,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
FECHA,,,,,,,,,,,,,,
2024-12-01 00:00:00,GRAVE,AV DE ANDALUCIA 19 C1,12,2024,0.00,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,441073.42,4468726.53
2024-12-01 00:00:00,GRAVE,C FUENTE CARRANTONA 44 C2,12,2024,0.00,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,446846.27,4474115.01
2024-12-01 00:00:00,GRAVE,AV DE ANDALUCIA 50 C2,12,2024,0.00,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,441090.00,4468177.59
2024-12-01 00:00:00,GRAVE,AV DE ARCENTALES 37 ACERA PAR,12,2024,0.00,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,448813.17,4476095.85
2024-12-01 00:00:00,GRAVE,AV DE LA ALBUFERA 393,12,2024,0.00,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,445996.27,4470763.74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-01 23:59:00,GRAVE,CALLE MAYOR 75,12,2024,23.59,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,439591.90,4474084.00
2024-12-01 23:59:00,GRAVE,CALLE ATOCHA 125,12,2024,23.59,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,441189.90,4473465.00
2024-12-01 23:59:00,GRAVE,CALLE GRAN VIA 71,12,2024,23.59,200.0,SI,0,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,439718.47,4474951.50


🔶 **Cesar Alejandro - Comentario:**
- La hora viene con un formato HH.MM en float, así que se aplica una fórmula matemática, multiplicando por 100 conseguimos un entero (se redondea para que no salga .99...). Al dividirlo por //100 devuelve los primeros dos valores, y al sacar el resto % 100 sacamos los minutos.
- Con pd.to_datetime construimos la fecha a partir de las columnas en un dataframe ficticio.
- El índice pasa a ser la fecha, con set_index borra automáticamente la columna fecha y la convierte en índice.

# ETAPA 2: Construcción del módulo 

El objetivo de esta etapa es realizar un paquete con clases y módulos, de forma que la respuesta a algunes de las consultas anteriores se pueda realizar con una llamada a métodos de una clase.

En esta sección se proporcionan algunas instrucciones para la creación del módulo `traficFines`.
La gran mayoría de métodos y funciones pedidas aquí, se han implementado de alguna manera en la sección anterior, por lo que la construcción del módulo y las clases que se pide a continuación es en gran medida una reorganización de código. El módulo `traficFines` debe exportar las clases descritas a continuación.

## Módulo `cache`

En muchos programas necesitamos descargar información de Internet para procesarla. En estos casos, es muy posible que se acceda a la misma url varias veces, lo que  consume tiempo y ancho de banda innecesarios. Además, durante el desarrollo,  puede ser más cómodo trabajar con una copia local ya descargada, en lugar de repetir la descarga. Para que estas tareas sean más sencillas vamos a crear un módulo llamado `cache` que tendrá 2 clases `Cache` y `CacheURL`. 
* La clase `Cache`, que gestiona el almacenamiento de ficheros descargados en un directorio local.

* La clase `CacheURL`, que amplía `Cache` para trabajar específicamente con URLs de Internet.


El módulo `cache` debe contener los siguientes elementos:

### Excepciones
* `CacheError` para cualquier tipo de situación de error que se pueda producir en el módulo.



### Clase `Cache`

Se encargará de guardar y recuperar datos en disco, de forma que estén disponibles incluso después de cerrar el programa.

La clase `Cache` va a almacenar cualquier tipo de fichero asociado con un identificador. Por defecto, los datos se almacenarán en el home del usuario, en un directorio para este ejercicio. Por ejemplo:
```
CACHE_DIR = Path.home() / ".my_cache"
```
Consultar [Path.home](https://docs.python.org/3/library/pathlib.html#pathlib.Path.home) del módulo `pathlib`.
Pero en el constructor se podría indicar que fuera cualquier otra carpeta del sistema operativo.

La clase `Cache` debe estar pensada para ser usada desde otras aplicaciones, no solo este ejercicio, por lo que cada objeto de la clase cache deberá tener los siguientes atributos:

* `app_name`, un `str` que será un nombre asociado a la aplicación concreta. Este atributo se usará para crear una subcarpeta dentro de la carpeta  `CACHE_DIR`. De esta forma, cada aplicación tendrá una carpeta dentro de `CACHE_DIR` para almacenar sus datos.
* `cache_dir`, un string con la ruta completa de la subcarpeta que usará la aplicación para almacenar sus datos.
* `obsolescence`, un `int` que indica el número de días que consideraremos que los datos de la caché siguen siendo válidos.

Estos atributos serán internos (privados) y solo podrán consultarse a través de propiedades (@property), nunca modificarse desde fuera.

El método __init__() tendrá a lo sumo dos parámetros: `app_name` y `obsolescence`.


Deberá tener (al menos) los siguientes métodos:

* `set(self, name:str, data:str) -> None` : método que almacena en `cache_dir` los datos en `data` con el nombre `name`.
* `exists(self, name: str) -> bool` : Comprueba si ya existe un fichero con el nombre name en la caché `cache_dir`.
* `load(self, name: str) -> str`: método que recupera los datos almacenados en la caché, con el nombre `name`. Si no hay ningún dato asociado deberá lanzar la excepción `CacheError`.
* `how_old(self, name: str) -> float`: método que nos dirá la edad en milisegundos del fichero almacenado en la caché con el nombre indicado. Esto sirve para comprobar si el fichero está caducado según el valor de `obsolescence`.
* `delete(self, name: str) -> None`: método que borra de la caché el fichero  con nombre `nane`. Si  no existe, no se lanza ninguna excepción.
* `clear(self) -> None`: método que borra todos los elementos de la caché.

Se pueden definir todos los métodos auxiliares que se deseen. Si se quiere indicar algún error no indicado, se debe lanzar la excepción `CacheError`.


### Clase `CacheUrl`

La clase `CacheURL` será una especialización de la clase `Cache` para trabajar específicamente con datos descargados de internet (URLs). 

Las URLs suelen ser muy largas y contener caracteres especiales (?, =, &, / …) que pueden dar problemas al usarlas directamente como nombres de ficheros en el sistema operativo.

El objetivo principal es poder reutilizar datos de una misma URL sin necesidad de descargarla varias veces, y hacerlo de una forma segura y portable en todos los sistemas operativos.

Para evitar problemas, no vamos a guardar la URL como nombre de fichero directamente, sino que guardaremos un hash de la URL. De esta forma, se obtiene un nombre de fichero corto y seguro.

Por ejemplo:

In [12]:
import hashlib
url = 'https://agenciatributaria.madrid.es/portal/site/contribuyente'
url_hashed = hashlib.md5(url.encode('utf-8')).hexdigest()
url_hashed

'456cb7e307cb1e8bb1e31528b2f8478b'

La clase `CacheURL` comparte el constructor con la clase `Cache` y los métodos que debe definir son:

* `get(self, url:str) -> str`
  Este método va a realizar un `get`  a la url indicada. Este método consultará si tiene ya un objeto asociado a esta petición, si no lo tiene realizará la petición correspondiente, guardará el resultado en la caché. Devolverá el objeto correspondiente.


Como el resultado depende de los argumentos dados, el nombre asociado al objeto que se guarda en la caché será un hash de la url junto con los parámetros opcionales a la petición.

Se deben redefinir los métodos de la clase de la forma adecuada:
* `exists(self, url: str, **kwargs) -> bool` 
* `load(self, url: str, **kwargs) -> str` 
* `how_old(self, name: str, **kwargs) -> float`
* `delete(self, name: str, **kwargs) -> None`


## Módulo `madridFines`

El módulo `madridFines` debe contener los siguientes elementos:

### Constantes

Puedes definir (al menos) la constante `RAIZ`.
```
RAIZ = "https://datos.madrid.es/"
```

### Excepciones
* `MadridError` para cualquier tipo de situación de error que se pueda producir en el módulo.

### Funciones del módulo

Se definen (al menos) las siguientes funciones:

* `get_url(year: int, month: int) -> str` . Función que recibe como argumentos un mes (1–12) y un año (con cuatro cifras, ej. 2024) y devuelva la URL del fichero CSV con los datos de multas correspondiente a ese mes y año, disponible en el portal de datos abiertos del Ayuntamiento de Madrid. Si el mes y año no están en la página se debe producir una excepción. Se recomienda resolver este ejercicio realizando scraping. Usar la librería `requests`  para hacer la petición HTTP y recuperar el HTML de la página. Usar `beautifulsoup4 (BS4)`  para parsear el HTML y localizar los enlaces correspondientes al año/mes buscado.


### Clase `MadridFines`

La clase `MadridFines` tiene como objetivo gestionar, limpiar, almacenar y analizar datos de multas de tráfico del Ayuntamiento de Madrid descargados desde el portal de datos abiertos. Es una clase pensada para automatizar el ciclo completo de análisis de multas: desde la descarga y preprocesamiento de multas de varios meses, hasta la generación de resúmenes y gráficos explicativos.

* Los objetos de la clase tienen tres atributos que se corresponden con:
  - `cacheurl`: un objeto de tipo `CacheURL` que permitirá gestionar la descarga de datos.
  - `data`: atributo para almacenar los datos (de tipo `pd.DataFrame`). Este atributo podrá contener multas de distintos meses.
  - `loaded`: un atributo de tipo lista de tuplas de la forma `(month, year)` para registrar los meses de datos contenidos en el atributo `data`.
    Los dos últimos atributos están relacionados, por lo que ha de garantizarse que su modificación no podrá realizarse desde fuera de la clase. 

* La clase ha de contener al menos los siguientes métodos:

  - `__init__`: método constructor. Acepta los argumentos `app_name` y `obsolescence` que se usarán para inicializar el atributo `cacheurl`. El valor inicial de los atributos `data` y `loaded` será el dataframe vacío y la lista vacía respectivamente.
 
  - `load(year: int, month: int, cacheurl) -> pd.DataFrame`: se trata de un método interno  y estático que utiliza el objeto cacheurl para acceder a los datos de multas del año `year` y mes `month`, a partir del cual se crea un dataframe de pandas que se será devuelto por la función.

  - `clean_cache() -> None`: método que borra todos los elementos de la caché.
 
  - `clean(df: pd.DataFrame) -> None`: método interno y estático que permite limpiar y normalizar los datos del dataframe `df`. La limpieza y preprocesamiento a realizar es la que se define en la sección `ETAPA 1` (elimina espacios en blanco en columnas de texto, convierte columnas numéricas (VEL_CIRCULA, VEL_LIMITE, COORDENADA-X, COORDENADA-Y), crea una columna fecha a partir de ANIO, MES y HORA, etc.). Es posible que no todos los meses tengan las mismas columnas. Por ejemplo, en Mayo de 2019, las columnas son COORDENADA_X, COORDENADA_Y, en lugar de  COORDENADA-X, COORDENADA-Y.
 
  - `add(self, year: int, month: int|None = None) -> None`: Permite añadir información de multas de un mes y un año concreto al dataset `data` actual. Evita duplicar meses ya cargados (si el mes ya existe, el método no hace nada). En caso contrario, se descargan los datos del año `year` y mes `month` usando el método `load`, se realiza la limpieza del dataframe descargado usando el método `clean` y se añade al dataset actual `data`. Este método tendrá que actualizar correctamente el atributo `loaded`. En el caso de que el parámetro `month` sea None, se descargarán los datos del año completo.
 
  - `fines_hour(self, fig_name: str)-> None`: método que, siguiendo el planteamiento de la consulta C1, genera un gráfico de líneas que muestra la evolución del número de multas a lo largo de las horas del día. Si hay varios meses cargados, dibuja una línea por cada mes. La función guarda el gráfico con nombre `fig_name` (por ejemplo `evolucion_multas.png`). Ejemplo:
    
    <div align="center">
    <img src="evolucion_multas05.png" alt="hora" style="width: 60%;">
    </div>
    
  - `fines_calification(self)-> pd.DataFrame`: analiza la distribución de multas por calificación, mes y año. Devuelve un DataFrame con el número total de multas distinguiendo por tipo para cada uno de los meses. El dataframe obtenido tendrá el siguiente aspecto:
    ```
        CALIFICACION   GRAVE    LEVE  MUY GRAVE
    MES ANIO                               
    5   2019       61124  111017        747
        2020       36752   43431        454
        2021       58935   97140        833
        2022      146864  102849        987
        2023      137561   94448       1092
    ```
  - `total_payment(self) -> pd.DataFrame`: método que genera un resumen con el importe total (máximo y mínimo) recaudado por mes y año. Seguir el razonamiento de las consultas C11 y C12. Una multa puede tener descuento por pronto pago del 50%, pero cabe la posibilidad de que el infractor no realice el pago en fecha para acojerse a dicho descuento.


## Tests

Define los tests que creas necesarios para comprobar que las clases funcionan correctamente. Los tests deberán estar en un directorio independiente, dentro del proyecto, pero no en el paquete que se distribuye. Es necesario que los tests tengan una cobertura cercana al 100% de los ficheros del paquete y que simulen los accesos a Internet y que los datos que potencialmente en el sistema de ficheros (por ejemplo el funcionamiento de la caché) se haga dentro de una carpeta data que se encuentre en el directorio de los tests.


# Entrega
La entrega consistirá en un fichero comprimido `zip` (no fichero `.rar`)  con lo siguiente:
- Un directorio con todos los ficheros del paquete. El directorio del paquete tiene que tener todo lo necesario para poder generar el fichero whl.
- El fichero whl para poder instalar el paquete.
- Fichero README.md con instrucciones de uso de la entrega.
- El presente cuaderno de jupyter con las consultas de la Etapa 1.
- Nuevo cuaderno de jupyter que contenga ejemplos e instrucciones para validar el correcto funcionamiento de las clases implementadas (y las excepciones correspondientes).

# Declaración responsable sobre autoría y uso ético de inteligencia artificial, otros recursos o ayuda externa

Con la entrega de esta tarea, declaro de manera responsable que
es el resultado de mi trabajo intelectual personal y creativo, y que ha sido elaborado de acuerdo con los principios éticos y las normas de integridad vigentes en la comunidad académica y, más
específicamente, en la [Universidad Complutense de Madrid](https://www.ucm.es/file/declaracion-responsable-sobre-autoria-y-uso-%C3%89tico-de-herramientas-de-ia).

Soy, pues, autor del material aquí incluido y, cuando no ha sido así y he tomado el material de
otra fuente, lo he citado o bien he declarado su procedencia de forma clara -incluidas, en su
caso, herramientas de inteligencia artificial-. Las ideas y aportaciones principales incluidas en este trabajo, y que acreditan la adquisición de competencias, son mías y no proceden de otras fuentes o han sido reescritas usando material de otras fuentes.

# Criterios de la corrección

Horas previstas para la realización de la práctica 15 horas.

* El cuaderno de jupyter con las consultas de la Etapa 1 (10%)
* Corrección de la solución con clases y paquetes: 40%
* Correcta documentación de los ficheros: 15%
* Correcta definición del paquete: 15%
* Correcta definición de tests: 20%


------------------